# Experiment 4: Decision Tree - ID3 Algorithm

## Overview
The ID3 (Iterative Dichotomiser 3) algorithm builds decision trees by recursively selecting attributes that best classify the training data. It uses Information Gain and Entropy to determine the best attribute to split on.

## Key Concepts
- **Entropy**: Measure of impurity/disorder in a dataset
- **Information Gain**: Reduction in entropy after a split
- **Best Split**: Attribute with highest information gain
- **Recursive Partitioning**: Continues until all examples are classified or no more attributes available

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

print("Decision Tree - ID3 Algorithm")
print("=" * 60)

## Manual ID3 Implementation

In [ ]:
class DecisionNode:
    """Node in the decision tree"""
    def __init__(self, attribute=None, value=None, results=None, true_branch=None, false_branch=None):
        self.attribute = attribute  # Attribute to test
        self.value = value  # Value of attribute to test
        self.results = results  # Dictionary of results if leaf node
        self.true_branch = true_branch  # Branch if test is true
        self.false_branch = false_branch  # Branch if test is false

def entropy(data):
    """
    Calculate entropy of a dataset
    Entropy = -Σ(p_i * log2(p_i)) where p_i is probability of class i
    """
    if len(data) == 0:
        return 0
    
    results = Counter(data)
    ent = 0.0
    for freq in results.values():
        p = freq / len(data)
        if p > 0:
            ent -= p * np.log2(p)
    return ent

def information_gain(data, attribute_idx, target_idx):
    """
    Calculate information gain for splitting on an attribute
    IG = Entropy(parent) - Σ((|subset|/|parent|) * Entropy(subset))
    """
    # Extract target values
    target_values = [row[target_idx] for row in data]
    parent_entropy = entropy(target_values)
    
    # Get unique values of the attribute
    attr_values = {}
    for row in data:
        val = row[attribute_idx]
        if val not in attr_values:
            attr_values[val] = []
        attr_values[val].append(row[target_idx])
    
    # Calculate weighted entropy of subsets
    weighted_entropy = 0.0
    total = len(data)
    for subset in attr_values.values():
        weight = len(subset) / total
        weighted_entropy += weight * entropy(subset)
    
    return parent_entropy - weighted_entropy

def split_data(data, attribute_idx, value):
    """Split data based on attribute value"""
    matching = [row for row in data if row[attribute_idx] == value]
    not_matching = [row for row in data if row[attribute_idx] != value]
    return matching, not_matching

def build_tree(data, attributes, target_idx, depth=0):
    """
    Recursively build decision tree using ID3 algorithm
    """
    target_values = [row[target_idx] for row in data]
    
    # Base case: all examples have same class
    if len(set(target_values)) == 1:
        return DecisionNode(results={target_values[0]: len(data)})
    
    # Base case: no more attributes
    if len(attributes) == 0:
        results = Counter(target_values)
        return DecisionNode(results=dict(results))
    
    # Find best attribute to split on
    best_gain = 0
    best_attr_idx = None
    
    for attr_idx in attributes:
        gain = information_gain(data, attr_idx, target_idx)
        if gain > best_gain:
            best_gain = gain
            best_attr_idx = attr_idx
    
    # If no good split found
    if best_attr_idx is None:
        results = Counter(target_values)
        return DecisionNode(results=dict(results))
    
    print(f"{'  ' * depth}Split on {attributes[best_attr_idx]} (Gain: {best_gain:.4f})")
    
    # Recursively build subtrees
    unique_values = set(row[best_attr_idx] for row in data)
    new_attributes = [a for a in attributes if a != best_attr_idx]
    
    node = DecisionNode(attribute=best_attr_idx, value=None)
    
    for val in unique_values:
        subset, _ = split_data(data, best_attr_idx, val)
        if len(subset) > 0:
            node.true_branch = build_tree(subset, new_attributes, target_idx, depth + 1)
            node.false_branch = build_tree(_, new_attributes, target_idx, depth + 1)
            break
    
    return node

print("Custom ID3 implementation loaded.")

## Using Sklearn Decision Tree (ID3-like)

In [ ]:
# Load iris dataset
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print("\nIris Dataset:")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Features: {feature_names}")
print(f"Classes: {target_names}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## Train Decision Tree

In [ ]:
# Build decision tree (entropy-based like ID3)
dt_classifier = DecisionTreeClassifier(
    criterion='entropy',  # ID3 uses entropy
    max_depth=4,
    random_state=42
)

dt_classifier.fit(X_train, y_train)

# Predictions
y_train_pred = dt_classifier.predict(X_train)
y_test_pred = dt_classifier.predict(X_test)

# Accuracy
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("\n" + "="*60)
print("DECISION TREE PERFORMANCE")
print("="*60)
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"\nTree Depth: {dt_classifier.get_depth()}")
print(f"Number of Leaves: {dt_classifier.get_n_leaves()}")

## Classification Report

In [ ]:
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred, target_names=target_names))

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_title('Confusion Matrix - Decision Tree Classifier')
plt.tight_layout()
plt.show()

## Tree Visualization

In [ ]:
# Visualize the decision tree
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_classifier, feature_names=feature_names, class_names=target_names,
          filled=True, ax=ax, rounded=True, fontsize=10)
plt.title("Decision Tree (ID3-based) for Iris Dataset")
plt.tight_layout()
plt.show()

print("Tree visualization complete.")

## New Sample Prediction

In [ ]:
# Test on new samples
new_samples = np.array([
    [5.0, 3.0, 1.5, 0.3],  # Likely Setosa
    [6.5, 3.0, 5.5, 1.8],  # Likely Virginica
    [5.9, 3.0, 4.2, 1.5]   # Likely Versicolor
])

predictions = dt_classifier.predict(new_samples)
probabilities = dt_classifier.predict_proba(new_samples)

print("\n" + "="*60)
print("NEW SAMPLE PREDICTIONS")
print("="*60)

for i, (sample, pred) in enumerate(zip(new_samples, predictions), 1):
    print(f"\nSample {i}: {sample}")
    print(f"Predicted Class: {target_names[pred]}")
    print(f"Probabilities:")
    for class_name, prob in zip(target_names, probabilities[i]):
        print(f"  {class_name}: {prob:.4f}")

## Feature Importance

In [ ]:
# Feature importance
importances = dt_classifier.feature_importances_

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(feature_names, importances, color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Feature Importance in Decision Tree')
ax.set_xlim(0, max(importances) * 1.1)

for i, (name, imp) in enumerate(zip(feature_names, importances)):
    ax.text(imp, i, f' {imp:.4f}', va='center')

plt.tight_layout()
plt.show()

print("\nFeature Importances:")
for name, imp in zip(feature_names, importances):
    print(f"  {name}: {imp:.4f}")

## Viva Questions

### Basic Concepts
1. **What is the ID3 algorithm and how does it work?**
   - Builds decision tree by recursively selecting best attribute to split on
   - Uses information gain and entropy to determine best split
   - Greedy algorithm that doesn't guarantee optimal tree

2. **Define Entropy in the context of decision trees.**
   - Measure of impurity or disorder in a dataset
   - Formula: Entropy = -Σ(p_i * log2(p_i))
   - Higher entropy means more mixed classes

3. **What is Information Gain?**
   - Reduction in entropy after split
   - Formula: IG = Entropy(parent) - Σ(weight * Entropy(child))
   - Attribute with highest gain is selected as split node

### Algorithm Questions
4. **Describe the ID3 algorithm step by step.**
   - Calculate entropy of current node
   - For each attribute: calculate information gain
   - Select attribute with highest gain
   - Recursively build subtrees for each attribute value
   - Stop when: all examples same class, no attributes left, or entropy is zero

5. **Why is ID3 considered a greedy algorithm?**
   - Makes locally optimal choice at each step
   - Selects attribute with best immediate gain
   - Does not consider impact on future splits
   - May not produce globally optimal tree

6. **How does the tree decide when to stop splitting?**
   - When all examples in node belong to same class
   - When no attributes remain to split on
   - When maximum depth is reached (in pruned versions)
   - When information gain becomes zero

### Practical Questions
7. **What are advantages of decision trees?**
   - Easy to understand and interpret
   - Requires minimal data preprocessing
   - Can handle both numerical and categorical data
   - Fast for making predictions

8. **What are limitations of decision trees?**
   - Prone to overfitting
   - Greedy nature doesn't guarantee optimal tree
   - Can be unstable with small changes in data
   - Biased toward attributes with many values

9. **How does ID3 differ from C4.5 algorithm?**
   - C4.5 uses gain ratio instead of information gain
   - C4.5 handles continuous attributes
   - C4.5 includes pruning mechanism
   - C4.5 can handle missing values

10. **What is tree pruning and why is it important?**
    - Removes branches that don't contribute to classification
    - Reduces overfitting
    - Improves generalization to new data
    - Makes tree simpler and faster